# 34. 해상도 거리 가중치 + 피처 제거

33번 위에서 두 가지를 바꾼다.

1. **숫자 컬럼 거리 가중치를 `log2(고유값 수)` 로 준다.** 탐색이 아니라 컬럼에서 직접 계산한다.
2. **피처를 뺀다.** 제거 단계마다 gamma 를 다시 고른다.

26번 시절 두 가지 모두 실패로 기록했는데 조건이 잘못돼 있었다.
거리 가중치는 Optuna 120 trial 로 골라 선택 편향에 걸렸고(30번, LB 전달률 0%),
피처 제거는 **gamma 를 2.0 에 고정하고 5-fold 로** 쟀다.

- 가중치는 **탐색하지 않는다.** 공식 하나뿐이라 고를 여지가 없다.
- 제거는 탐색이므로 **무작위 대조**로 검정한다.
- 평가는 **20-fold**. 5-fold 는 학습 2400행이라 검증 행의 쌍둥이가 학습 폴드에 80% 만
  들어가서 효과가 압축된다. 20-fold 는 2850행으로 95% 라 최종 모델(3000행)에 가깝다.

## 1. 설정

SVR·`mean_working` 폴백·하드 스위치 게이트는 33번과 같다.

In [1]:
import warnings
import numpy as np
import pandas as pd
from sklearn.compose import TransformedTargetRegressor
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import mean_absolute_error as mae
from sklearn.model_selection import KFold
from sklearn.preprocessing import OneHotEncoder, QuantileTransformer, RobustScaler
from sklearn.svm import SVR

warnings.filterwarnings('ignore')
RS = 42
CUT, THR = 0.026, 0.04
SEARCH_SEED = 42                 # 탐색 전용
VSEEDS = [2024, 7, 123]          # 탐색에 쓰지 않은 검증 시드

NUM = ['age', 'height', 'weight', 'cholesterol', 'systolic_blood_pressure',
       'diastolic_blood_pressure', 'glucose', 'bone_density']
CAT = ['gender', 'activity', 'smoke_status', 'medical_history',
       'family_medical_history', 'sleep_pattern', 'edu_level']

train = pd.read_csv('../data/train.csv')
test = pd.read_csv('../data/test.csv')
y = train['stress_score'].to_numpy(float)
LVL = train['mean_working'].round(0).fillna(-1).to_numpy(float)
LVL_TEST = test['mean_working'].round(0).fillna(-1).to_numpy(float)
MEDIAN = float(np.median(y))


def numeric(df):
    x = df[NUM].copy()
    x['bmi'] = (df['weight'] / (df['height'] / 100) ** 2).round(2)
    return x


NUMCOLS = list(NUM) + ['bmi']
sc = RobustScaler().fit(numeric(train).to_numpy(float))
ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False,
                    dtype=float).fit(train[CAT].fillna('Unknown'))
NUNIQ = numeric(train).nunique().to_numpy(float)

NUMZ, NUMZ_T = (sc.transform(numeric(d).to_numpy(float)) for d in (train, test))
CATZ, CATZ_T = (ohe.transform(d[CAT].fillna('Unknown')) for d in (train, test))

CAT_SLICE, _o = {}, 0
for _c, _cats in zip(CAT, ohe.categories_):
    CAT_SLICE[_c] = slice(_o, _o + len(_cats)); _o += len(_cats)


def iso_table(lv_tr, y_tr, lv_t):
    m = lv_tr >= 0
    ir = IsotonicRegression(out_of_bounds='clip').fit(lv_tr[m], y_tr[m])
    return np.where(lv_t >= 0, ir.predict(np.where(lv_t >= 0, lv_t, 0)),
                    y_tr.mean()).astype(float)


def gate(ps, fb):
    return np.where((np.abs(ps - MEDIAN) < CUT) & (np.abs(fb - MEDIAN) > THR), fb, ps)


def score(X, nfold, seed, g):
    ps = np.zeros(len(y)); fb = np.zeros(len(y))
    for t, v in KFold(nfold, shuffle=True, random_state=seed).split(X):
        m = TransformedTargetRegressor(
            regressor=SVR(C=4.0, gamma=g, kernel='rbf', epsilon=0.0),
            transformer=QuantileTransformer(output_distribution='normal',
                                            n_quantiles=1000, random_state=RS))
        ps[v] = np.clip(m.fit(X[t], y[t]).predict(X[v]), 0, 1)
        fb[v] = iso_table(LVL[t], y[t], LVL[v])
    return mae(y, gate(ps, fb))


print(f'train {train.shape}  숫자 {len(NUMCOLS)}개  범주 {len(CAT)}블록 {CATZ.shape[1]}컬럼')

train (3000, 18)  숫자 9개  범주 7블록 23컬럼


## 2. 해상도 거리 가중치

컬럼마다 값의 해상도가 다르다. 고유값이 2000개인 컬럼과 60개인 컬럼을 같은 비중으로 거리에
넣으면 해상도 낮은 컬럼의 양자화 오차가 거리를 지배한다.

`w = log2(고유값 수)`, 평균 1 로 정규화. 적합도 탐색도 없다.

In [2]:
def build(num_keep, cat_keep, weighted=True, numz=None, catz=None):
    """남은 숫자 컬럼으로 가중치를 재정규화해 블록을 합친다."""
    numz = NUMZ if numz is None else numz
    catz = CATZ if catz is None else catz
    ni = [NUMCOLS.index(c) for c in num_keep]
    if weighted:
        w = np.log2(NUNIQ[ni]); w = w / w.mean()
    else:
        w = np.ones(len(ni))
    return np.hstack([numz[:, ni] * w] + [catz[:, CAT_SLICE[c]] for c in cat_keep])


W_FULL = np.log2(NUNIQ); W_FULL = W_FULL / W_FULL.mean()
print(pd.DataFrame({'고유값 수': NUNIQ.astype(int), 'log2': np.log2(NUNIQ).round(2),
                    '가중치': W_FULL.round(3)}, index=NUMCOLS).to_string())

                          고유값 수   log2    가중치
age                          73   6.19  0.691
height                     1828  10.84  1.210
weight                     1986  10.96  1.223
cholesterol                2215  11.11  1.241
systolic_blood_pressure      89   6.48  0.723
diastolic_blood_pressure     61   5.93  0.662
glucose                    2107  11.04  1.233
bone_density                200   7.64  0.854
bmi                        1367  10.42  1.163


### 순열 귀무검정

가중치 9개를 컬럼 사이에 섞는다. 값의 다중집합은 그대로고 배정만 무작위다.
배정에 정보가 없으면 실제 가중치가 순열들 사이에 묻혀야 한다. 5-fold, 시드 3개.

In [3]:
PSEEDS = [42, 2024, 7]
X_plain = np.hstack([NUMZ, CATZ])


def perm_score(w):
    X = np.hstack([NUMZ * w, CATZ])
    return np.mean([score(X, 5, s, 2.0) for s in PSEEDS])


base_p = np.mean([score(X_plain, 5, s, 2.0) for s in PSEEDS])
real_p = perm_score(W_FULL) - base_p
rng = np.random.default_rng(0)
perm_d = np.array([perm_score(rng.permutation(W_FULL)) - base_p for _ in range(12)])

print(pd.Series(perm_d, index=[f'순열 {i+1}' for i in range(12)]).round(6).to_string())
print(f'\n순열 평균 {perm_d.mean():+.6f}, σ {perm_d.std(ddof=1):.6f}, 최선 {perm_d.min():+.6f}')
print(f'실제 가중치 {real_p:+.6f}')
print(f'z = {(real_p - perm_d.mean()) / perm_d.std(ddof=1):+.2f}σ, '
      f'순열 대비 승 {(perm_d > real_p).sum()}/12')

순열 1     0.000210
순열 2     0.000003
순열 3    -0.000008
순열 4     0.000202
순열 5     0.000327
순열 6     0.000071
순열 7     0.000133
순열 8     0.000011
순열 9    -0.000070
순열 10   -0.000225
순열 11    0.000022
순열 12    0.000269

순열 평균 +0.000079, σ 0.000156, 최선 -0.000225
실제 가중치 -0.000235
z = -2.00σ, 순열 대비 승 12/12


배정을 섞으면 이득이 사라진다. 27번 Optuna 가중치가 30번에서 기각된 것과 다른 점은
**여기엔 trial 이 없다**는 것이다. 공식 하나에서 나온 값이라 고를 여지가 없다.

## 3. Backward elimination

숫자 9개와 범주 7블록을 후보로 하나씩 빼본다. 후보마다 gamma 를 `{2.0, 2.8}` 에서 다시 고른다.
차원이 줄면 거리 스케일이 바뀌므로 gamma 를 고정하면 제거가 항상 손해로 나온다.

탐색은 10-fold·시드 42 만 쓴다. 검증 시드는 건드리지 않는다.

In [4]:
GAMMAS = [2.0, 2.8]


def best_gamma(num_keep, cat_keep, nfold=10, seed=SEARCH_SEED):
    X = build(num_keep, cat_keep)
    return min((score(X, nfold, seed, g), g) for g in GAMMAS)


num_keep, cat_keep = NUMCOLS.copy(), CAT.copy()
base, gbest = best_gamma(num_keep, cat_keep)
print(f'시작  10-fold MAE {base:.6f} (g={gbest})\n')
path = []

for rnd in range(1, 5):
    cands = []
    for c in num_keep:
        s, g = best_gamma([x for x in num_keep if x != c], cat_keep)
        cands.append((s, g, 'num', c))
    for c in cat_keep:
        s, g = best_gamma(num_keep, [x for x in cat_keep if x != c])
        cands.append((s, g, 'cat', c))
    cands.sort()
    print(f'[라운드 {rnd}] 기준 {base:.6f}')
    for cs, cg, ck, cc in cands[:4]:
        print(f'   {ck}:{cc:<26} {cs:.6f} (g={cg})  Δ {cs - base:+.6f}')
    s, g, kind, c = cands[0]
    if s >= base:
        print('   -> 개선 없음, 종료\n')
        break
    if kind == 'num':
        num_keep = [x for x in num_keep if x != c]
    else:
        cat_keep = [x for x in cat_keep if x != c]
    path.append((kind, c)); base, gbest = s, g
    print(f'   -> 제거 {kind}:{c}   새 기준 {base:.6f} (g={gbest})\n')

print(f'제거: {path}')
print(f'숫자 {num_keep}')
print(f'범주 {cat_keep}')
print(f'gamma {gbest}')

시작  10-fold MAE 0.129720 (g=2.8)



[라운드 1] 기준 0.129720
   cat:smoke_status               0.129464 (g=2.8)  Δ -0.000256
   cat:sleep_pattern              0.129539 (g=2.8)  Δ -0.000181
   num:diastolic_blood_pressure   0.129655 (g=2.8)  Δ -0.000065
   cat:family_medical_history     0.129699 (g=2.8)  Δ -0.000022
   -> 제거 cat:smoke_status   새 기준 0.129464 (g=2.8)



[라운드 2] 기준 0.129464
   cat:sleep_pattern              0.129287 (g=2.8)  Δ -0.000177
   num:height                     0.129360 (g=2.0)  Δ -0.000103
   num:bmi                        0.129432 (g=2.0)  Δ -0.000032
   num:diastolic_blood_pressure   0.129436 (g=2.8)  Δ -0.000027
   -> 제거 cat:sleep_pattern   새 기준 0.129287 (g=2.8)



[라운드 3] 기준 0.129287
   num:bmi                        0.129237 (g=2.8)  Δ -0.000049
   cat:family_medical_history     0.129340 (g=2.8)  Δ +0.000053
   num:age                        0.129354 (g=2.8)  Δ +0.000068
   num:bone_density               0.129461 (g=2.8)  Δ +0.000174
   -> 제거 num:bmi   새 기준 0.129237 (g=2.8)



[라운드 4] 기준 0.129237
   num:age                        0.129390 (g=2.8)  Δ +0.000152
   num:bone_density               0.129478 (g=2.8)  Δ +0.000240
   cat:family_medical_history     0.129504 (g=2.8)  Δ +0.000266
   num:systolic_blood_pressure    0.129824 (g=2.8)  Δ +0.000587
   -> 개선 없음, 종료

제거: [('cat', 'smoke_status'), ('cat', 'sleep_pattern'), ('num', 'bmi')]
숫자 ['age', 'height', 'weight', 'cholesterol', 'systolic_blood_pressure', 'diastolic_blood_pressure', 'glucose', 'bone_density']
범주 ['gender', 'activity', 'medical_history', 'family_medical_history', 'edu_level']
gamma 2.8


`bmi` 제거는 원리적으로 설명된다. `weight/(height/100)^2` 라 height·weight 가 이미 있으면
새 정보가 없고 체격 부분공간만 3개 컬럼으로 부풀린다.

범주 두 블록은 사후 설명이 없다. 무작위 대조가 필요하다.

## 4. 무작위 대조

40회쯤 탐색해 고른 조합이라 선택 편향이 있다. 같은 개수(숫자 1 + 범주 2)를 무작위로 뺀
12개와 비교한다. 20-fold, 검증 시드 3개.

In [5]:
def eval20(num_drop, cat_drop, g=None):
    g = gbest if g is None else g
    X = build([c for c in NUMCOLS if c not in num_drop],
              [c for c in CAT if c not in cat_drop])
    return np.mean([score(X, 20, s, g) for s in VSEEDS])


full = eval20([], [])
sel = eval20([c for k, c in path if k == 'num'], [c for k, c in path if k == 'cat'])
print(f'전체 피처 (g={gbest})  {full:.6f}')
print(f'선택 제거 조합        {sel:.6f}   Δ {sel - full:+.6f}\n')

rng = np.random.default_rng(0)
seen = {tuple(sorted(c for _, c in path))}
nulls = []
while len(nulls) < 12:
    nd, cd = rng.choice(NUMCOLS, 1, replace=False), rng.choice(CAT, 2, replace=False)
    key = tuple(sorted([*nd, *cd]))
    if key in seen:
        continue
    seen.add(key)
    nulls.append((eval20(list(nd), list(cd)) - full, f'{nd[0]} + {cd[0]},{cd[1]}'))
    print(f'  무작위 {len(nulls):2d}  {nulls[-1][0]:+.6f}   ({nulls[-1][1]})')

d = np.array([x[0] for x in nulls])
print(f'\n무작위 평균 {d.mean():+.6f}, σ {d.std(ddof=1):.6f}, 최선 {d.min():+.6f}')
print(f'선택 조합   {sel - full:+.6f}')
print(f'z = {(sel - full - d.mean()) / d.std(ddof=1):+.2f}σ, '
      f'무작위 대비 승 {(d > sel - full).sum()}/12')

전체 피처 (g=2.8)  0.125379
선택 제거 조합        0.125120   Δ -0.000260



  무작위  1  +0.001692   (bone_density + edu_level,medical_history)


  무작위  2  +0.000438   (weight + edu_level,gender)


  무작위  3  +0.001207   (height + family_medical_history,edu_level)


  무작위  4  +0.001649   (systolic_blood_pressure + medical_history,edu_level)


...

  무작위  8  +0.000975   (glucose + sleep_pattern,family_medical_history)


  무작위  9  -0.000142   (age + sleep_pattern,gender)


  무작위 10  +0.000800   (age + medical_history,activity)


  무작위 11  +0.001171   (cholesterol + edu_level,gender)


  무작위 12  +0.000781   (age + family_medical_history,medical_history)

무작위 평균 +0.001009, σ 0.000579, 최선 -0.000142
선택 조합   -0.000260
z = -2.19σ, 무작위 대비 승 12/12


무작위 제거는 평균적으로 **나빠진다.** 피처를 함부로 빼면 손해라는 뜻이고,
그 분포 안에서 선택 조합만 개선 쪽에 있다.

## 5. 33번과 비교

두 변경을 따로 본다. 같은 시드·같은 폴드, 20-fold.

In [6]:
X33 = build(NUMCOLS, CAT, weighted=False)
X_w = build(NUMCOLS, CAT)
X_final = build(num_keep, cat_keep)

rows = []
for s in VSEEDS:
    rows.append({'시드': s,
                 '33번 (w=1, g=2.0)': score(X33, 20, s, 2.0),
                 '가중치만 (g=2.0)': score(X_w, 20, s, 2.0),
                 f'34번 (가중치+제거, g={gbest})': score(X_final, 20, s, gbest)})

cmp = pd.DataFrame(rows)
print(cmp.round(6).to_string(index=False))
m = cmp.drop(columns='시드').mean()
print('\n평균:')
print(m.round(6).to_string())
for c in m.index[1:]:
    print(f'{c} : 33번 대비 {m[c] - m.iloc[0]:+.6f}')

  시드  33번 (w=1, g=2.0)  가중치만 (g=2.0)  34번 (가중치+제거, g=2.8)
2024          0.125286      0.124924             0.124614
   7          0.126524      0.126196             0.125772
 123          0.125632      0.125226             0.124973

평균:
33번 (w=1, g=2.0)       0.125814
가중치만 (g=2.0)           0.125449
34번 (가중치+제거, g=2.8)    0.125120
가중치만 (g=2.0) : 33번 대비 -0.000365
34번 (가중치+제거, g=2.8) : 33번 대비 -0.000695


## 6. 최종 학습 및 제출

In [7]:
X = build(num_keep, cat_keep)
X_test = build(num_keep, cat_keep, numz=NUMZ_T, catz=CATZ_T)

final = TransformedTargetRegressor(
    regressor=SVR(C=4.0, gamma=gbest, kernel='rbf', epsilon=0.0),
    transformer=QuantileTransformer(output_distribution='normal',
                                    n_quantiles=1000, random_state=RS))
pred_svr = np.clip(final.fit(X, y).predict(X_test), 0, 1)
pred_fb = iso_table(LVL, y, LVL_TEST)
pred = gate(pred_svr, pred_fb)

use = (np.abs(pred_svr - MEDIAN) < CUT) & (np.abs(pred_fb - MEDIAN) > THR)
prev = pd.read_csv('../submissions/submit_33_hard_switch.csv')['stress_score'].to_numpy()
print(f'피처 {X.shape[1]}컬럼 (33번 32컬럼), gamma {gbest}')
print(f'폴백 교체 행 {int(use.sum())}개')
print(f'33번 대비 바뀐 행 {(np.abs(pred - prev) > 1e-9).sum()}, '
      f'이동 평균 {np.abs(pred - prev).mean():.4f}')
print(f'예측 평균 {pred.mean():.4f}, 표준편차 {pred.std():.4f}, '
      f'범위 {pred.min():.3f}~{pred.max():.3f}')

sub = pd.read_csv('../data/sample_submission.csv')
sub['stress_score'] = pred
sub.to_csv('../submissions/submit_34_feature_elim.csv', index=False)
print('\nsaved -> submissions/submit_34_feature_elim.csv')

피처 25컬럼 (33번 32컬럼), gamma 2.8
폴백 교체 행 170개
33번 대비 바뀐 행 592, 이동 평균 0.0049
예측 평균 0.5003, 표준편차 0.2044, 범위 0.000~1.000

saved -> submissions/submit_34_feature_elim.csv


## 7. 정리

| 항목 | 33번 | 34번 |
|---|---|---|
| 숫자 거리 가중치 | 전부 1 | `log2(고유값 수)` 정규화 |
| 피처 | 9숫자 + 7범주 (32컬럼) | 8숫자 + 5범주 (25컬럼) |
| gamma | 2.0 | 2.8 |
| 주 평가 | 5-fold | 20-fold |

26번·30번에서 두 축을 모두 닫았다고 기록했는데 조건이 잘못돼 있었다.
가중치는 120 trial 중 최선을 고르는 방식이라 선택 편향에 걸렸고(여기선 공식 하나만 쓴다),
피처 제거는 gamma 고정 + 5-fold 라 손해로만 나왔다.

두 변경 모두 귀무 대조로 검정했고 탐색에 쓰지 않은 시드로만 평가했다.

### 규정 관련

- 스케일러·인코더·등위회귀·SVR 전부 train(또는 학습 폴드)에서만 적합했다
- 거리 가중치는 train 컬럼의 고유값 수에서 계산했다. test 를 보지 않는다
- 근접 중복행 탐색, 최근접이웃 매칭, test 행 간 정보 공유 코드가 없다
- 게이트 두 조건은 모델 자기 출력과 train 레벨 통계만 쓴다